In [0]:
# Service Principal credentials
application_id = "**********************"
authentication_key = "********************"
tenant_id = "*****************************"

# Set Spark config for ADLS access
spark.conf.set("fs.azure.account.auth.type.delearn.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.delearn.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.delearn.dfs.core.windows.net", application_id)
spark.conf.set("fs.azure.account.oauth2.client.secret.delearn.dfs.core.windows.net", authentication_key)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.delearn.dfs.core.windows.net", "https://login.microsoftonline.com/" + tenant_id + "/oauth2/token")

print("✅ Spark config set!")

✅ Spark config set!


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =========================================
# 1. RECEIVE CURRENT CDC FILE PATH FROM ADF
# =========================================
dbutils.widgets.text("cdc_batch_path", "")
cdc_file_path = dbutils.widgets.get("cdc_batch_path")

# =========================================
# 2. TARGET SILVER PATH
# =========================================
silver_path = "abfss://cdcsilver@delearn.dfs.core.windows.net/customer/"

print("CDC file path received from ADF:")
print(cdc_file_path)

print("Silver path:")
print(silver_path)

# =========================================
# 3. READ CURRENT CDC FILE
# =========================================
df_cdc = spark.read.parquet(cdc_file_path)

print("Raw CDC Data:")
display(df_cdc)

# =========================================
# 4. DEBUG: CHECK DUPLICATES IN CDC BATCH
# =========================================
print("Duplicate CustomerIDs in this batch:")
display(
    df_cdc.groupBy("CustomerID")
          .count()
          .filter("count > 1")
          .orderBy("CustomerID")
)

# =========================================
# 5. DEDUPLICATE CDC BATCH
#    Keep only latest row per CustomerID
# =========================================
op_priority = (
    F.when(F.col("OperationType") == "DELETE", F.lit(3))
     .when(F.col("OperationType") == "UPDATE", F.lit(2))
     .when(F.col("OperationType") == "INSERT", F.lit(1))
     .otherwise(F.lit(0))
)

ordering_cols = []

if "ChangeTime" in df_cdc.columns:
    ordering_cols.append(F.col("ChangeTime").desc())

if "UpdatedAt" in df_cdc.columns:
    ordering_cols.append(F.col("UpdatedAt").desc())

if "SeqVal" in df_cdc.columns:
    ordering_cols.append(F.col("SeqVal").desc())

ordering_cols.append(op_priority.desc())

window_spec = Window.partitionBy("CustomerID").orderBy(*ordering_cols)

df_latest = (
    df_cdc
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

print("Deduplicated CDC Data (latest row per CustomerID):")
display(df_latest)

# =========================================
# 6. LOAD TARGET DELTA TABLE
# =========================================
target = DeltaTable.forPath(spark, silver_path)
target_df = spark.read.format("delta").load(silver_path)

target_columns = target_df.columns
source_columns = df_latest.columns

print("Target columns:")
print(target_columns)

print("Source columns after dedup:")
print(source_columns)

# =========================================
# 7. PREPARE COLUMN LISTS
# =========================================
# Exclude technical soft delete columns
business_columns = [c for c in target_columns if c not in ["IsDeleted", "DeletedAt"]]

# Keep only columns that exist in both source CDC and target Silver
common_columns = [c for c in business_columns if c in source_columns]

print("Common columns available in CDC file:")
print(common_columns)

# =========================================
# 8. BUILD UPDATE MAP
# =========================================
# Update only columns that are really present in CDC file
update_set = {c: f"s.`{c}`" for c in common_columns}

# For normal updates, keep row active
update_set["IsDeleted"] = "false"
update_set["DeletedAt"] = "null"

print("Update mapping:")
print(update_set)

# =========================================
# 9. BUILD INSERT MAP
# =========================================
insert_values = {}

for c in target_columns:
    if c in common_columns:
        insert_values[c] = f"s.`{c}`"
    elif c == "IsDeleted":
        insert_values[c] = "false"
    elif c == "DeletedAt":
        insert_values[c] = "null"
    else:
        insert_values[c] = "null"

print("Insert mapping:")
print(insert_values)

# =========================================
# 10. APPLY SOFT DELETE MERGE
# =========================================
(
    target.alias("t")
    .merge(
        df_latest.alias("s"),
        "t.CustomerID = s.CustomerID"
    )
    # UPDATE existing rows
    .whenMatchedUpdate(
        condition="s.OperationType = 'UPDATE'",
        set=update_set
    )
    # SOFT DELETE existing rows
    .whenMatchedUpdate(
        condition="s.OperationType = 'DELETE'",
        set={
            "IsDeleted": "true",
            "DeletedAt": "current_timestamp()"
        }
    )
    # INSERT new rows
    .whenNotMatchedInsert(
        condition="s.OperationType IN ('INSERT', 'UPDATE')",
        values=insert_values
    )
    .execute()
)

print("Soft Delete CDC merge completed successfully.")

# =========================================
# 11. VERIFY FINAL SILVER
# =========================================
print("Final Silver Data:")
display(spark.read.format("delta").load(silver_path))

CDC file path received from ADF:
abfss://cdcbronze@delearn.dfs.core.windows.net/customer_cdc/2026-06-12T14-52-49.parquet
Silver path:
abfss://cdcsilver@delearn.dfs.core.windows.net/customer/
Raw CDC Data:


OperationType,CustomerID,CustomerName,City,Email,UpdatedAt,ChangeTime,StartLSN,SeqVal,OperationCode,BatchFromLSN,BatchToLSN,IngestionTime
INSERT,1,Customer_1,Pune,customer1@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E00003,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z
INSERT,2,Customer_2,Bangalore,customer2@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E00005,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z
INSERT,3,Customer_3,Chennai,customer3@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E00007,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z
INSERT,4,Customer_4,Mumbai,customer4@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E00009,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z
INSERT,5,Customer_5,Hyderabad,customer5@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E0000B,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z
INSERT,6,Customer_6,Pune,customer6@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E0000D,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z
INSERT,7,Customer_7,Bangalore,customer7@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E0000F,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z
INSERT,8,Customer_8,Chennai,customer8@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E00011,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z
INSERT,9,Customer_9,Mumbai,customer9@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E00013,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z
INSERT,10,Customer_10,Hyderabad,customer10@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E00015,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z


Duplicate CustomerIDs in this batch:


CustomerID,count
10,2
15,2
20,2
25,2
30,2
35,2
40,2
45,2
50,2
55,2


Deduplicated CDC Data (latest row per CustomerID):


OperationType,CustomerID,CustomerName,City,Email,UpdatedAt,ChangeTime,StartLSN,SeqVal,OperationCode,BatchFromLSN,BatchToLSN,IngestionTime
INSERT,1,Customer_1,Pune,customer1@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E00003,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z
INSERT,2,Customer_2,Bangalore,customer2@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E00005,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z
INSERT,3,Customer_3,Chennai,customer3@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E00007,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z
INSERT,4,Customer_4,Mumbai,customer4@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E00009,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z
INSERT,5,Customer_5,Hyderabad,customer5@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E0000B,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z
INSERT,6,Customer_6,Pune,customer6@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E0000D,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z
INSERT,7,Customer_7,Bangalore,customer7@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E0000F,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z
INSERT,8,Customer_8,Chennai,customer8@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E00011,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z
INSERT,9,Customer_9,Mumbai,customer9@gmail.com,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.607Z,0x0000004700000A580042,0x00000047000009E00013,2,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z
UPDATE,10,Customer_10,Chennai,customer10@gmail.com,2026-06-12T14:51:04.253158Z,2026-06-12T14:51:04.25Z,0x0000004900000CF00016,0x0000004900000CF0000F,4,0x0000004700000A580042,0x0000004900000D08000D,2026-06-12T14:53:29.073605Z


Target columns:
['CustomerID', 'CustomerName', 'Email', 'PhoneNumber', 'City', 'State', 'Country', 'ZipCode', 'Age', 'Gender', 'CustomerType', 'AnnualIncome', 'AccountBalance', 'CreditScore', 'Status', 'JoinDate', 'LastLoginDate', 'LastTransactionDate', 'CreatedDate', 'CreatedBy', 'UpdatedAt', 'UpdatedBy', 'IsDeleted', 'DeletedAt']
Source columns after dedup:
['OperationType', 'CustomerID', 'CustomerName', 'City', 'Email', 'UpdatedAt', 'ChangeTime', 'StartLSN', 'SeqVal', 'OperationCode', 'BatchFromLSN', 'BatchToLSN', 'IngestionTime']
Common columns available in CDC file:
['CustomerID', 'CustomerName', 'Email', 'City', 'UpdatedAt']
Update mapping:
{'CustomerID': 's.`CustomerID`', 'CustomerName': 's.`CustomerName`', 'Email': 's.`Email`', 'City': 's.`City`', 'UpdatedAt': 's.`UpdatedAt`', 'IsDeleted': 'false', 'DeletedAt': 'null'}
Insert mapping:
{'CustomerID': 's.`CustomerID`', 'CustomerName': 's.`CustomerName`', 'Email': 's.`Email`', 'PhoneNumber': 'null', 'City': 's.`City`', 'State': 'n

CustomerID,CustomerName,Email,PhoneNumber,City,State,Country,ZipCode,Age,Gender,CustomerType,AnnualIncome,AccountBalance,CreditScore,Status,JoinDate,LastLoginDate,LastTransactionDate,CreatedDate,CreatedBy,UpdatedAt,UpdatedBy,IsDeleted,DeletedAt
1,Customer_1,customer1@gmail.com,9000000001,Pune,Maharashtra,India,500001,21,Female,Regular,310000.00,5500.00,601,Active,2026-06-11,2026-06-11T14:03:19.601733Z,2026-06-11T14:03:19.601733Z,2026-06-12T14:03:19.601733Z,System,2026-06-12T14:03:19.601733Z,System,false,null
2,Customer_2,customer2@gmail.com,9000000002,Bangalore,Karnataka,India,500002,22,Male,VIP,320000.00,6000.00,602,Active,2026-06-10,2026-06-10T14:03:19.601733Z,2026-06-10T14:03:19.601733Z,2026-06-12T14:03:19.601733Z,System,2026-06-12T14:03:19.601733Z,System,false,null
3,Customer_3,customer3@gmail.com,9000000003,Chennai,Tamil Nadu,India,500003,23,Female,Premium,330000.00,6500.00,603,Active,2026-06-09,2026-06-09T14:03:19.601733Z,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.601733Z,System,2026-06-12T14:03:19.601733Z,System,false,null
4,Customer_4,customer4@gmail.com,9000000004,Mumbai,Maharashtra,India,500004,24,Male,Regular,340000.00,7000.00,604,Active,2026-06-08,2026-06-08T14:03:19.601733Z,2026-06-11T14:03:19.601733Z,2026-06-12T14:03:19.601733Z,System,2026-06-12T14:03:19.601733Z,System,false,null
5,Customer_5,customer5@gmail.com,9000000005,Hyderabad,Telangana,India,500005,25,Female,VIP,350000.00,7500.00,605,Active,2026-06-07,2026-06-07T14:03:19.601733Z,2026-06-10T14:03:19.601733Z,2026-06-12T14:03:19.601733Z,System,2026-06-12T14:03:19.601733Z,System,false,null
6,Customer_6,customer6@gmail.com,9000000006,Pune,Maharashtra,India,500006,26,Male,Premium,360000.00,8000.00,606,Active,2026-06-06,2026-06-06T14:03:19.601733Z,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.601733Z,System,2026-06-12T14:03:19.601733Z,System,false,null
7,Customer_7,customer7@gmail.com,9000000007,Bangalore,Karnataka,India,500007,27,Female,Regular,370000.00,8500.00,607,Active,2026-06-05,2026-06-12T14:03:19.601733Z,2026-06-11T14:03:19.601733Z,2026-06-12T14:03:19.601733Z,System,2026-06-12T14:03:19.601733Z,System,false,null
8,Customer_8,customer8@gmail.com,9000000008,Chennai,Tamil Nadu,India,500008,28,Male,VIP,380000.00,9000.00,608,Active,2026-06-04,2026-06-11T14:03:19.601733Z,2026-06-10T14:03:19.601733Z,2026-06-12T14:03:19.601733Z,System,2026-06-12T14:03:19.601733Z,System,false,null
9,Customer_9,customer9@gmail.com,9000000009,Mumbai,Maharashtra,India,500009,29,Female,Premium,390000.00,9500.00,609,Active,2026-06-03,2026-06-10T14:03:19.601733Z,2026-06-12T14:03:19.601733Z,2026-06-12T14:03:19.601733Z,System,2026-06-12T14:03:19.601733Z,System,false,null
11,Customer_11,customer11@gmail.com,9000000011,Pune,Maharashtra,India,500011,31,Female,VIP,410000.00,10500.00,611,Active,2026-06-01,2026-06-08T14:03:19.601733Z,2026-06-10T14:03:19.601733Z,2026-06-12T14:03:19.601733Z,System,2026-06-12T14:03:19.601733Z,System,false,null


In [0]:
df_silver=spark.read.format("delta") \
    .load("abfss://cdcsilver@delearn.dfs.core.windows.net/customer/") \
    .display()
    

CustomerID,CustomerName,Email,PhoneNumber,City,State,Country,ZipCode,Age,Gender,CustomerType,AnnualIncome,AccountBalance,CreditScore,Status,JoinDate,LastLoginDate,LastTransactionDate,CreatedDate,CreatedBy,UpdatedAt,UpdatedBy,IsDeleted,DeletedAt
1,Customer_1,customer1@gmail.com,9000000001,Pune,Maharashtra,India,500001,21,Female,Regular,310000.00,5500.00,601,Active,2026-06-11,2026-06-11T16:09:27.38237Z,2026-06-11T16:09:27.38237Z,2026-06-12T16:09:27.38237Z,System,2026-06-12T16:09:27.38237Z,System,false,null
2,Customer_2,customer2@gmail.com,9000000002,Bangalore,Karnataka,India,500002,22,Male,VIP,320000.00,6000.00,602,Active,2026-06-10,2026-06-10T16:09:27.38237Z,2026-06-10T16:09:27.38237Z,2026-06-12T16:09:27.38237Z,System,2026-06-12T16:09:27.38237Z,System,false,null
3,Customer_3,customer3@gmail.com,9000000003,Chennai,Tamil Nadu,India,500003,23,Female,Premium,330000.00,6500.00,603,Active,2026-06-09,2026-06-09T16:09:27.38237Z,2026-06-12T16:09:27.38237Z,2026-06-12T16:09:27.38237Z,System,2026-06-12T16:09:27.38237Z,System,false,null
4,Customer_4,customer4@gmail.com,9000000004,Mumbai,Maharashtra,India,500004,24,Male,Regular,340000.00,7000.00,604,Active,2026-06-08,2026-06-08T16:09:27.38237Z,2026-06-11T16:09:27.38237Z,2026-06-12T16:09:27.38237Z,System,2026-06-12T16:09:27.38237Z,System,false,null
5,Customer_5,customer5@gmail.com,9000000005,Hyderabad,Telangana,India,500005,25,Female,VIP,350000.00,7500.00,605,Active,2026-06-07,2026-06-07T16:09:27.38237Z,2026-06-10T16:09:27.38237Z,2026-06-12T16:09:27.38237Z,System,2026-06-12T16:09:27.38237Z,System,false,null
6,Customer_6,customer6@gmail.com,9000000006,Pune,Maharashtra,India,500006,26,Male,Premium,360000.00,8000.00,606,Active,2026-06-06,2026-06-06T16:09:27.38237Z,2026-06-12T16:09:27.38237Z,2026-06-12T16:09:27.38237Z,System,2026-06-12T16:09:27.38237Z,System,false,null
7,Customer_7,customer7@gmail.com,9000000007,Bangalore,Karnataka,India,500007,27,Female,Regular,370000.00,8500.00,607,Active,2026-06-05,2026-06-12T16:09:27.38237Z,2026-06-11T16:09:27.38237Z,2026-06-12T16:09:27.38237Z,System,2026-06-12T16:09:27.38237Z,System,false,null
8,Customer_8,customer8@gmail.com,9000000008,Chennai,Tamil Nadu,India,500008,28,Male,VIP,380000.00,9000.00,608,Active,2026-06-04,2026-06-11T16:09:27.38237Z,2026-06-10T16:09:27.38237Z,2026-06-12T16:09:27.38237Z,System,2026-06-12T16:09:27.38237Z,System,false,null
9,Customer_9,customer9@gmail.com,9000000009,Mumbai,Maharashtra,India,500009,29,Female,Premium,390000.00,9500.00,609,Active,2026-06-03,2026-06-10T16:09:27.38237Z,2026-06-12T16:09:27.38237Z,2026-06-12T16:09:27.38237Z,System,2026-06-12T16:09:27.38237Z,System,false,null
11,Customer_11,customer11@gmail.com,9000000011,Pune,Maharashtra,India,500011,31,Female,VIP,410000.00,10500.00,611,Active,2026-06-01,2026-06-08T16:09:27.38237Z,2026-06-10T16:09:27.38237Z,2026-06-12T16:09:27.38237Z,System,2026-06-12T16:09:27.38237Z,System,false,null
